In [41]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

In [42]:
df = pd.read_csv('/kaggle/input/datasets/salikhussaini49/prediction-of-sepsis/Dataset.csv')

print(df.shape)
df.head()

(1552210, 44)


,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,1,0,17072
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,2,0,17072
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,3,0,17072
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,4,0,17072
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,...,NaN,330.0,68.54,0,NaN,NaN,-0.02,5,0,17072


In [43]:
df = df.sort_values(by=['Patient_ID', 'ICULOS'])

In [44]:
vital_signs = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
vital_signs = [c for c in vital_signs if c in df.columns]

exclude_cols = ['SepsisLabel', 'Patient_ID', 'ICULOS']
lab_features = [c for c in df.columns if c not in vital_signs + exclude_cols]

In [46]:
for col in vital_signs + lab_features:
    if col in df.columns:
        df[col + '_missing'] = df[col].isnull().astype(int)

In [47]:
cols = vital_signs + lab_features

df[cols] = df.groupby('Patient_ID')[cols].ffill()
df[cols] = df.groupby('Patient_ID')[cols].bfill()

In [48]:
agg_dict = {}

for col in vital_signs + lab_features:
    agg_dict[col] = ['mean', 'max', 'min', 'last']

for col in [c for c in df.columns if '_missing' in c]:
    agg_dict[col] = 'sum'

# target
agg_dict['SepsisLabel'] = 'max'

df_patient = df.groupby('Patient_ID').agg(agg_dict)

In [49]:
df_patient.columns = ['_'.join(col).strip() for col in df_patient.columns.values]
df_patient = df_patient.reset_index()

In [50]:
X = df_patient.drop(columns=['Patient_ID', 'SepsisLabel_max'])
y = df_patient['SepsisLabel_max']

In [51]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [52]:
imputer = SimpleImputer(strategy='median')

X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

In [53]:
scaler = StandardScaler()

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [56]:
X_train.shape

(32268, 205)

In [57]:
y_train.shape

(32268,)

In [58]:
from imblearn.over_sampling import SMOTE

print(f"Before SMOTE - Train: {y_train.value_counts().to_dict()}")

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"After SMOTE - Train: {pd.Series(y_train_resampled).value_counts().to_dict()}")

X_train = X_train_resampled
y_train = y_train_resampled

Before SMOTE - Train: {0: 29922, 1: 2346}
After SMOTE - Train: {0: 29922, 1: 29922}


In [59]:
X_train.shape

(59844, 205)

In [60]:
print("Train missing:", X_train.isnull().sum().sum())
print("Test missing:", X_test.isnull().sum().sum())

Train missing: 0
Test missing: 0


In [61]:
df_patient.head()

,Patient_ID,HR_mean,HR_max,HR_min,HR_last,O2Sat_mean,O2Sat_max,O2Sat_min,O2Sat_last,Temp_mean,...,PTT_missing_sum,WBC_missing_sum,Fibrinogen_missing_sum,Platelets_missing_sum,Age_missing_sum,Gender_missing_sum,Unit1_missing_sum,Unit2_missing_sum,HospAdmTime_missing_sum,SepsisLabel_max
0,1,102.055556,117.0,76.0,84.0,91.398148,100.0,85.0,85.0,36.636296,...,54,52,54,52,0,0,54,54,0,0
1,2,60.956522,94.0,54.0,55.0,97.086957,100.0,94.0,95.0,36.181739,...,23,22,23,22,0,0,0,0,0,0
2,3,79.927083,93.0,68.0,78.0,95.333333,99.0,91.0,97.0,37.469583,...,46,45,48,45,0,0,0,0,0,0
3,4,102.672414,113.0,93.0,108.0,98.155172,100.0,95.5,98.0,36.447931,...,27,28,29,27,0,0,0,0,0,0
4,5,76.604167,88.0,61.0,80.0,97.677083,99.0,96.0,97.0,37.072292,...,46,45,48,45,0,0,0,0,0,0


In [62]:
df_patient['Patient_ID'].nunique()

40336

In [63]:
X_train.isnull().sum().sum()

np.int64(0)

In [64]:
df_patient.shape

(40336, 207)

In [65]:
# Combine X and y back (optional but useful)
train_df = X_train.copy()
train_df['SepsisLabel'] = y_train.values

test_df = X_test.copy()
test_df['SepsisLabel'] = y_test.values


# Save to CSV
train_df.to_csv('train_dataset.csv', index=False)
test_df.to_csv('test_dataset.csv', index=False)

print("Train and test datasets saved successfully!")

Train and test datasets saved successfully!


In [66]:
train_df.to_pickle('train_final.pkl')
test_df.to_pickle('test_final.pkl')